# CLP Transformer Validation Pipeline

This notebook demonstrates how to load a trained Transformer model and evaluate its performance on benchmark instances. It compares the model-guided policies against the standard VCS heuristic baseline using both Greedy and Beam Search solving strategies.

Loads the trained weights and architecture configuration for the `CLPTransformer`.

## 1. Model Loading

In [2]:
from models.clp_transformer import CLPTransformer
from training.training import load_model

model_name = "M30"
model = load_model(CLPTransformer, model_name)

## 2. Input Adapter Configuration

Initializes the adapter responsible for vectorizing the environment's current state. This ensures the raw geometric and action data during the evaluation phase is properly formatted to match the model's expected input dimensions.

In [1]:
# Configure the adapter for model input
from data.adapters.input.v1 import InputAdapterV1

input_adapter = InputAdapterV1(max_blocks=10000, max_pblocks=64, max_actions=64)

## 3. Greedy Search Evaluation

Evaluates and compares the single-path execution strategies on a benchmark instance. It contrasts the pure heuristic baseline (`VCSSolver`) against the model-driven policy (`GreedyModelSolver`). The `w` parameter controls the candidate pool size (a beam width of 8 generates 64 candidates), and `min_fr` controls the block generation compactness.

In [3]:
from solvers.greedy import GreedyModelSolver, VCSSolver

instance_file = "benchmarks/BR8.txt"
instance_number = 0
min_fr = 0.98
w = 8 # W = 8 selects 64 candidates

greedy_solver = GreedyModelSolver(model, w=w, input_adapter=input_adapter, min_fr=min_fr)
greedy_eval, _ = greedy_solver.solve(instance_file, instance_number)

vcs_solver = VCSSolver(min_fr=min_fr)
vcs_eval, _ = vcs_solver.solve(instance_file, instance_number)

print("GreedyModel:", f'{greedy_eval:.2f}')
print("VCSSolver:", f'{vcs_eval:.2f}')

GreedyModel: 93.85
VCSSolver: 91.47


## 4. Beam Search Evaluation

Evaluates the advanced tree-search solving strategies. This cell compares the pure heuristic approach (`BSG_VCS_Solver`) against two model-assisted variants: `BSG_ModelVCS` (which uses the model for node expansion and the heuristic for evaluation rollouts) and `BSG_ModelFull` (which uses the model for both expansion and evaluation rollouts).

In [4]:
from solvers.beam_search import BSG_VCS_Solver, BSG_ModelFull_Solver, BSG_ModelVCS_Solver

instance_file = "benchmarks/BR8.txt"
instance_number = 0
min_fr = 0.98
w = 8

bsg_vcs_solver = BSG_VCS_Solver(w=w, min_fr=min_fr)
bsg_vcs_eval, _ = bsg_vcs_solver.solve(instance_file, instance_number)

bsg_model_full_solver = BSG_ModelFull_Solver(model, input_adapter, w=w, min_fr=min_fr)
bsg_model_full_eval, _ = bsg_model_full_solver.solve(instance_file, instance_number)

bsg_model_vcs_solver = BSG_ModelVCS_Solver(model, input_adapter, w=w, min_fr=min_fr)
bsg_model_vcs_eval, _ = bsg_model_vcs_solver.solve(instance_file, instance_number)

print("BSG VCS:", f'{bsg_vcs_eval:.2f}')
print("BSG ModelFull:", f'{bsg_model_full_eval:.2f}')
print("BSG ModelVCS:", f'{bsg_model_vcs_eval:.2f}')

BSG VCS: 95.05
BSG ModelFull: 96.08
BSG ModelVCS: 95.67
